<a href="https://colab.research.google.com/github/honordold/DS1001-LABS-Projects/blob/main/2026_10_16_%E2%80%94_Walmart_Warm_Up_SKU_Mess_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [1]:
import pandas as pd
PATH = 'data/product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [2]:
pop = cat['product_name'].str.lower().str.contains('pop') # Find all the product_name rows that contain pop
print(f"Count: {pop.sum()}") # Print count
cat[pop] # Matching rows

Count: 4


,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [3]:
straw = cat[cat['product_name'].str.contains('pop', case=False, na=False) # Find all the strawberries
            & cat['product_name'].str.contains('straw', case=False, na=False)]

display(straw[['sku', 'product_name']].drop_duplicates())
not_straw = cat[cat['product_name'].str.contains('pop', case=False, na=False) # Find the pop's that don't contrain straw
                & ~cat['product_name'].str.contains('straw', case=False, na=False)]

display(not_straw[['sku', 'product_name']].drop_duplicates())


,sku,product_name
0,PT-12,pop-tarts strawberry
1,SKU#459812,Strawberry Pop-Tart
2,POPTART-STRAW,Pop-Tarts Strawberry
3,PT_STRAWBERRY,pop-tarts strawberry


,sku,product_name


**Pop-Tart rows that are a different product:** There are no pop-tart rows that aren't strawberry since my dataframe came back empty.

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [4]:
# Strawberry Pop-Tart variants -> one canonical SKU
CANON = {
    'POPTART-STRAW': 'POPTART-STRAW',
    'PT-12': 'POPTART-STRAW',
    'SKU#459812': 'POPTART-STRAW',
    'PT_STRAWBERRY': 'POPTART-STRAW',
}

def consolidate(df):
    out = df.copy()  # leave original untouched
    out['canonical_sku'] = out['sku'].map(CANON).fillna(out['sku'])  # unmapped SKUs keep their own value
    return out

consolidated = consolidate(cat)

assert len(consolidated) == len(cat)  # no rows dropped
assert consolidated['canonical_sku'].notna().all()  # no NaNs
print("Distinct SKUs:", cat['sku'].nunique(), "->", consolidated['canonical_sku'].nunique())  # expect 45 -> 42

Distinct SKUs: 45 -> 42


### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [5]:
print("Before:", consolidated['category'].nunique(), "distinct")
display(consolidated['category'].value_counts())

consolidated['category'] = consolidated['category'].str.strip().str.title()  # remove extra spaces, fix case

print("After:", consolidated['category'].nunique(), "distinct")

# Remaining duplicate labels -> one category name
CATEGORY_MAP = {
    'Breakfast Foods': 'Breakfast',
    'Emergency Supplies': 'Emergency',
    'Groceries': 'Grocery',
    'Alcoholic Beverages': 'Alcohol',
    'Baby Products': 'Baby',
}

consolidated['category'] = consolidated['category'].replace(CATEGORY_MAP)  # unmapped categories stay as-is

print("After merging:", consolidated['category'].nunique(), "distinct")  # expect 10
display(consolidated['category'].value_counts())

Before: 27 distinct


,count
category,
Emergency,5
Emergency Supplies,3
EMERGENCY,3
ENTERTAINMENT,3
GROCERY,3
Breakfast,2
entertainment,2
household,2
ELECTRONICS,2


After: 15 distinct
After merging: 10 distinct


,count
category,
Emergency,12
Grocery,6
Entertainment,5
Breakfast,4
Electronics,4
Baby,4
Household,4
Alcohol,2
Snacks,2


### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [6]:
# Normalize names: lowercase, punctuation -> spaces, sort the words
cat['name_key'] = (cat['product_name'].str.lower()
                   .str.replace(r'[^a-z0-9 ]', ' ', regex=True)
                   .str.split()
                   .apply(lambda words: ' '.join(sorted(words))))

# Keep only keys shared by more than one SKU
candidates = cat.groupby('name_key').filter(lambda g: g['sku'].nunique() > 1)
candidates.sort_values('name_key')[['name_key', 'sku', 'product_name']]

,name_key,sku,product_name
12,24 beer light pack,BEER-24,Light Beer (24-pack)
13,24 beer light pack,LTBEER24,Light Beer (24-pack)
8,4 aa batteries pack,BAT-AA4,AA Batteries 4-Pack
9,4 aa batteries pack,BATT_AA_4PK,AA Batteries 4-Pack
30,aid first kit,FIRSTAID-KIT,First Aid Kit
29,aid first kit,FAID-01,First-Aid Kit
15,board game monopoly,MONOPOLY-01,Monopoly Board Game
14,board game monopoly,BG-MONO,Monopoly Board Game
16,board game monopoly,GAME-MONO,Board Game - Monopoly
24,bread white,WHITEBREAD,WHITE BREAD


### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [7]:
assert consolidated['canonical_sku'].notna().all(), 'mapping produced nulls in canonical_sku'
assert len(consolidated) == len(cat), 'consolidation changed the row count'
print('Checks passed: no nulls,', len(consolidated), 'rows before and after')

Checks passed: no nulls, 45 rows before and after


### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

In [8]:
monopoly = cat[cat['product_name'].str.contains('monopoly', case=False, na=False)] # Take all the monopoly product names
monopoly[['sku', 'product_name', 'package_size', 'unit_cost']] # Print their details

,sku,product_name,package_size,unit_cost
14,BG-MONO,Monopoly Board Game,1pk,9.96
15,MONOPOLY-01,Monopoly Board Game,24-pack,12.51
16,GAME-MONO,Board Game - Monopoly,1pk,NaN


*a) Two product families that are fragmented are AA batteries and monopoly. AA batteries have three SKUs: BATT-100, BAT-AA4, and BATT_AA_4PK. Monopoly also has 3 SKUs: BG-MONO, MONOPOLY-01, and GAME-MONO. For both of them I would create one canonical SKU, such as MONOPOLY-01.*

*b) Consolidation stops being a data-cleaning task and becomes a business decision is when the data can't prove two items aer the same, so merging depends on whether a customer or buyer would treat the products as interchangeable. One example is with moonopoly. BG-MONO is listed a 1pk while MONOPOLY-01 is listed as a 24-pk. If you merge them, one unit of MONOPOLY-01 might actually be 24 games, so the units sold could be wrong. On the other hand, 24-pk could also be an error. I would ask the category manager or buyer of toys if these were the same product. I could also ask the inventory manager if 24-pack means one case, and if a 1-pack also means one case.*

